# Solution: Honey Production — Linear Regression

**Course context:** Codecademy *Data Science Path — Linear Regression* (Honey Production project) extended.

**Goal:** Investigate the long-term decline in U.S. honey production using ordinary least-squares linear regression, project future production, explore robustness, and communicate findings for different audiences.

**Data:** `data/honeyproduction.csv` — state-level panel (1998–2012) with columns:  
`state`, `numcol`, `yieldpercol`, `totalprod`, `stocks`, `priceperlb`, `prodvalue`, `year`.

---

## Quick Cheat Sheet (keep this open while coding)

| Task | Pandas / NumPy / sklearn |
|------|--------------------------|
| Load | `pd.read_csv("data/honeyproduction.csv")` |
| Yearly mean | `df.groupby("year")["totalprod"].mean().reset_index()` |
| Feature matrix | `X = prod["year"].values.reshape(-1, 1)` |
| Target | `y = prod["totalprod"].values` |
| Model | `regr = LinearRegression(); regr.fit(X, y)` |
| Slope / intercept | `regr.coef_[0]`, `regr.intercept_` |
| In-sample preds | `y_pred = regr.predict(X)` |
| Future years | `X_fut = np.arange(2013, 2051).reshape(-1, 1)` |
| R² | `regr.score(X, y)` |
| Pure-NumPy slope | `m = np.polyfit(years, y, 1)[0]` or closed-form |
| Closed-form | `m = Σ((x-x̄)(y-ȳ)) / Σ((x-x̄)²)` , `b = ȳ - m·x̄` |

**Key numbers from this solution:** slope ≈ −88 300 lbs/year, R² ≈ 0.58, 2050 forecast ≈ 186 k lbs (near-zero ~2052).

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## 1. Check out the Data

Load the CSV and inspect structure with `.head()`, `.shape`, `.describe()`, and unique years.

In [ ]:
df = pd.read_csv("data/honeyproduction.csv")
print("Shape:", df.shape)
print("Years:", sorted(df["year"].unique()))
display(df.head())
display(df.describe().round(2))

## 2. Mean total production per year

We care about the national trend → aggregate with `groupby("year")["totalprod"].mean()`.

In [ ]:
prod_per_year = df.groupby("year")["totalprod"].mean().reset_index()
display(prod_per_year)

## 3–4. Create feature matrix X and target y

sklearn expects a 2-D array for X: reshape with `.values.reshape(-1, 1)`.

In [ ]:
X = prod_per_year["year"].values.reshape(-1, 1)
y = prod_per_year["totalprod"].values
print("X shape:", X.shape, "y shape:", y.shape)

## 5. Scatterplot of total production vs year

Is there a vaguely linear (downward) relationship?

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="#F4A261", s=60, edgecolor="k")
plt.xlabel("Year")
plt.ylabel("Mean Total Production (lbs)")
plt.title("US Honey Production by Year (state averages)")
plt.ticklabel_format(style="sci", axis="y", scilimits=(6, 6))
plt.tight_layout()
plt.show()

## 6–8. Create, fit, and inspect the LinearRegression model

In [ ]:
regr = LinearRegression()
regr.fit(X, y)

print(f"Slope (coef_):     {regr.coef_[0]:.2f} lbs per year")
print(f"Intercept:         {regr.intercept_:.2f}")
print(f"R² (score):        {regr.score(X, y):.4f}")

## 9–10. In-sample predictions and overlay the fitted line

In [ ]:
y_predict = regr.predict(X)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, color="#F4A261", s=60, edgecolor="k", label="Observed mean")
plt.plot(X, y_predict, color="#2A9D8F", lw=2.5, label="Linear fit")
plt.xlabel("Year")
plt.ylabel("Mean Total Production (lbs)")
plt.title("Observed Trend + OLS Fit")
plt.legend()
plt.ticklabel_format(style="sci", axis="y", scilimits=(6, 6))
plt.tight_layout()
plt.show()

## 11–13. Predict the honey decline into the future (2013–2050)

In [ ]:
X_future = np.array(range(2013, 2051)).reshape(-1, 1)
future_predict = regr.predict(X_future)

print(f"Predicted mean production in 2050: {future_predict[-1]:,.0f} lbs")
print(f"Year when fitted line crosses zero: {(0 - regr.intercept_) / regr.coef_[0]:.1f}")

plt.figure(figsize=(9, 5))
plt.scatter(X, y, color="#F4A261", s=50, edgecolor="k", zorder=3, label="Observed")
plt.plot(X, y_predict, color="#2A9D8F", lw=2, label="Fitted line")
plt.plot(X_future, future_predict, color="#E76F51", lw=2.5, ls="--", label="Projection 2013–2050")
plt.axhline(0, color="gray", ls=":", alpha=0.7)
plt.scatter([2050], [future_predict[-1]], color="#E76F51", s=80, zorder=4)
plt.xlabel("Year")
plt.ylabel("Mean Total Production (lbs)")
plt.title("Projected Decline in US Honey Production")
plt.legend(loc="upper right")
plt.ticklabel_format(style="sci", axis="y", scilimits=(6, 6))
plt.tight_layout()
plt.show()

---
## Alternate Implementations (same result, different tools)

### A. NumPy `polyfit` (degree-1 polynomial)

In [ ]:
years = prod_per_year["year"].values
m_poly, b_poly = np.polyfit(years, y, 1)
print(f"polyfit slope: {m_poly:.2f}, intercept: {b_poly:.2f}")
assert np.isclose(m_poly, regr.coef_[0], atol=1e-6)
assert np.isclose(b_poly, regr.intercept_, atol=1e-4)

### B. Closed-form ordinary least squares

In [ ]:
x = years.astype(float)
x_bar, y_bar = x.mean(), y.mean()
m_closed = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
b_closed = y_bar - m_closed * x_bar
print(f"Closed-form slope: {m_closed:.2f}, intercept: {b_closed:.2f}")
assert np.isclose(m_closed, regr.coef_[0], atol=1e-6)

### C. Pure-Python loop (educational, slow for large n)

In [ ]:
def ols_slope_intercept(x_list, y_list):
    n = len(x_list)
    x_mean = sum(x_list) / n
    y_mean = sum(y_list) / n
    num = sum((x_list[i] - x_mean) * (y_list[i] - y_mean) for i in range(n))
    den = sum((x_list[i] - x_mean) ** 2 for i in range(n))
    m = num / den
    b = y_mean - m * x_mean
    return m, b

m_py, b_py = ols_slope_intercept(list(years), list(y))
print(f"Pure-Python slope: {m_py:.2f}, intercept: {b_py:.2f}")

---
## More Practice

### Practice 1 — Yield per colony trend
Repeat the analysis for `yieldpercol` (average honey yield per colony). Is the slope steeper or flatter than total production?

In [ ]:
yield_per_year = df.groupby("year")["yieldpercol"].mean().reset_index()
X_y = yield_per_year["year"].values.reshape(-1, 1)
y_y = yield_per_year["yieldpercol"].values
regr_y = LinearRegression().fit(X_y, y_y)
print(f"Yield slope: {regr_y.coef_[0]:.3f} lbs/colony per year  |  R² = {regr_y.score(X_y, y_y):.3f}")

plt.figure(figsize=(7, 4))
plt.scatter(X_y, y_y, color="#264653", s=50)
plt.plot(X_y, regr_y.predict(X_y), color="#E76F51", lw=2)
plt.xlabel("Year"); plt.ylabel("Mean yield per colony (lbs)")
plt.title("Yield per Colony Trend")
plt.tight_layout(); plt.show()

### Practice 2 — Price per pound
Does the real price of honey rise as production falls? Fit `priceperlb` vs year.

In [ ]:
price_per_year = df.groupby("year")["priceperlb"].mean().reset_index()
X_p = price_per_year["year"].values.reshape(-1, 1)
y_p = price_per_year["priceperlb"].values
regr_p = LinearRegression().fit(X_p, y_p)
print(f"Price slope: ${regr_p.coef_[0]:.4f} per year  |  R² = {regr_p.score(X_p, y_p):.3f}")

plt.figure(figsize=(7, 4))
plt.scatter(X_p, y_p, color="#2A9D8F", s=50)
plt.plot(X_p, regr_p.predict(X_p), color="#E9C46A", lw=2)
plt.xlabel("Year"); plt.ylabel("Mean price per lb ($)")
plt.title("Honey Price Trend")
plt.tight_layout(); plt.show()

### Practice 3 — Cross-section: totalprod vs numcol (ignore year)
Treat every state-year as an observation. Does colony count strongly predict total production?

In [ ]:
X_c = df["numcol"].values.reshape(-1, 1)
y_c = df["totalprod"].values
regr_c = LinearRegression().fit(X_c, y_c)
print(f"Colonies → production slope: {regr_c.coef_[0]:.2f} lbs per colony  |  R² = {regr_c.score(X_c, y_c):.3f}")

---
## Simulation Section — Change a few values, observe different results

### Simulation 1: Additive noise sensitivity of the slope

In [ ]:
np.random.seed(42)
n_sims = 300
noise_frac = 0.15          # ← change this (0.05, 0.25, …)
slopes = []
for _ in range(n_sims):
    noise = np.random.normal(0, y.std() * noise_frac, size=len(y))
    m = LinearRegression().fit(X, y + noise).coef_[0]
    slopes.append(m)

print(f"Original slope: {regr.coef_[0]:.0f}")
print(f"Sim mean slope: {np.mean(slopes):.0f}  ± {np.std(slopes):.0f} (sd)")
print(f"95% interval:   [{np.percentile(slopes, 2.5):.0f}, {np.percentile(slopes, 97.5):.0f}]")

plt.figure(figsize=(7, 4))
plt.hist(slopes, bins=30, color="#264653", edgecolor="white", alpha=0.85)
plt.axvline(regr.coef_[0], color="#E76F51", lw=2.5, label="Original")
plt.xlabel("Estimated slope (lbs/year)"); plt.ylabel("Count")
plt.title(f"Monte-Carlo slopes under {noise_frac*100:.0f}% noise")
plt.legend(); plt.tight_layout(); plt.show()

### Simulation 2: Sample-size / year-window effect
What happens if we only had the last 8 years of data?

In [ ]:
window = 8                 # ← change this
recent = prod_per_year.tail(window)
Xr = recent["year"].values.reshape(-1, 1)
yr = recent["totalprod"].values
regr_r = LinearRegression().fit(Xr, yr)
print(f"Full-series slope:   {regr.coef_[0]:.0f}")
print(f"Last-{window}-yr slope: {regr_r.coef_[0]:.0f}")
print(f"2050 forecast (recent only): {regr_r.predict([[2050]])[0]:,.0f}")

### Simulation 3: Bootstrap confidence interval for the 2050 prediction

In [ ]:
np.random.seed(7)
n_boot = 500
preds_2050 = []
idx = np.arange(len(y))
for _ in range(n_boot):
    sample = np.random.choice(idx, size=len(idx), replace=True)
    mb = LinearRegression().fit(X[sample], y[sample])
    preds_2050.append(mb.predict([[2050]])[0])

lo, hi = np.percentile(preds_2050, [2.5, 97.5])
print(f"2050 point forecast: {future_predict[-1]:,.0f}")
print(f"Bootstrap 95% CI:    [{lo:,.0f}, {hi:,.0f}]")

---
## Audience-Adapted Interpretation Notes

*(Drawn from the supplied audience-analysis guidance: data literacy, subject knowledge, Experts / Technicians / Executives / Nonspecialists.)*

### For data-literate / technical supervisors
- Report slope (−88 303 lbs yr⁻¹), intercept, R² ≈ 0.58, residual diagnostics, and the bootstrap CI.
- Emphasize that the linear extrapolation is a **descriptive** trend, not a causal model of colony collapse disorder, pesticides, or habitat loss.
- Offer the closed-form / polyfit verification and the noise-sensitivity simulation as quality-control evidence.

### For executives / policy decision-makers
- Headline: “Under the 1998–2012 trend, average state honey production is projected near zero by the early 2050s.”
- One chart (the future-projection figure) + the single number for 2050 is usually enough.
- Avoid jargon; translate R² into “the linear trend accounts for roughly 58 % of the year-to-year variation.”

### For subject-matter experts (apiculturists, agronomists)
- They already know CCD, Varroa, almonds, etc. Skip the biology primer.
- They will care whether the decline is driven more by fewer colonies (`numcol`) or lower yield per colony (`yieldpercol`) — hence the extra practice models.
- Waterfall or contribution charts (change in colonies × yield) can be more familiar than a plain scatter.

### For nonspecialists / general public
- Use simple language: “On average, U.S. states produced about 88 000 fewer pounds of honey each year.”
- Prefer bar/line charts over residual plots; highlight the 2050 story with a clear annotation.
- Avoid abbreviations (CCD, OLS) unless defined on first use.

### Mixed-audience tip
Put the technical appendix (code, diagnostics, bootstrap) after a short executive summary so each reader can stop at the depth they need.

---
## Key Takeaways

1. U.S. state-average honey production declined roughly **88 300 lbs per year** from 1998–2012 (R² ≈ 0.58).
2. A naïve linear extrapolation reaches near-zero production around **2052** and predicts only ~186 000 lbs in 2050.
3. The same qualitative decline appears in **yield per colony**; price per pound has risen, consistent with scarcity.
4. Results are moderately sensitive to noise and to the exact year window; always report uncertainty (bootstrap CI).
5. Communication must be adapted: technical detail for analysts, a single clear chart + headline number for executives, plain language for the public.